# CNN Baseline using Mel Spectrograms

## Introduction

Classical Machine Learning models use compact MFCC summary vectors. These features are efficient, but they discard much of the original time-frequency structure of the audio signal.

Convolutional Neural Networks can operate directly on two-dimensional Mel Spectrogram representations. A Mel Spectrogram preserves how spectral energy changes over time while using a perceptual frequency scale that better matches human hearing.

Mel Spectrogram generation is handled in `03_feature_extraction.ipynb` because it is a DSP feature extraction step. This notebook only loads the saved Mel tensors and trains the CNN baseline.

The dataset strategy remains unchanged: GTZAN and FMA Medium are already combined and split into train / validation / test sets.

## Objectives

The main goals of this notebook are:

- load saved Mel Spectrogram tensors and labels,
- encode labels consistently,
- create the CNN model from `src/models/cnn_model.py`,
- train using reusable helpers from `src/training/train.py`,
- evaluate using reusable helpers from `src/training/evaluate.py`,
- save the trained model and evaluation artifacts.

Macro F1 remains an important metric because the combined dataset is imbalanced. Strong performance on frequent genres does not necessarily imply good performance on minority genres.

## Load Configuration

All Mel tensor paths, CNN hyperparameters, and CNN artifact paths are imported from `src/utils/config.py`.

The notebook uses PyTorch because the project already depends on `torch` and `torchaudio`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

from src.data.feature_extraction import load_mel_dataset
from src.models.cnn_model import create_cnn_model
from src.training.evaluate import evaluate_torch_model
from src.training.train import (
    save_model,
    save_torch_model,
    train_torch_model,
)
from src.utils.config import *

## Load Saved Mel Spectrogram Features

Notebook 03 saves Mel Spectrogram tensors and matching label arrays under `data/processed/mel/`.

This notebook expects those files to already exist. If they are missing, run `03_feature_extraction.ipynb` first.

In [ ]:
X_train, labels_train = load_mel_dataset(
    MEL_TRAIN_PATH,
    MEL_TRAIN_LABELS_PATH
)

X_validation, labels_validation = load_mel_dataset(
    MEL_VALIDATION_PATH,
    MEL_VALIDATION_LABELS_PATH
)

X_test, labels_test = load_mel_dataset(
    MEL_TEST_PATH,
    MEL_TEST_LABELS_PATH
)

print("Train tensors:", X_train.shape)
print("Validation tensors:", X_validation.shape)
print("Test tensors:", X_test.shape)
print("Expected CNN input shape:", CNN_INPUT_SHAPE)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## Prepare Labels

The label encoder is fitted on the training labels and reused for validation and test labels. This keeps class indices consistent across splits.

In [ ]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(labels_train)
y_validation = label_encoder.transform(labels_validation)
y_test = label_encoder.transform(labels_test)

class_names = list(label_encoder.classes_)

print("Classes:", class_names)
pd.Series(labels_train).value_counts().loc[class_names]

## CNN Model Architecture

The CNN architecture is defined in `src/models/cnn_model.py`.

It is intentionally small because this is a baseline model:

- three convolutional blocks,
- max pooling in the early blocks,
- adaptive global average pooling,
- dense classifier with dropout,
- softmax-equivalent multiclass output through `CrossEntropyLoss` during training.

This model uses richer time-frequency input than MFCC vectors, but training is slower because it operates on full two-dimensional Mel Spectrogram tensors.

In [ ]:
torch.manual_seed(CNN_RANDOM_STATE)
np.random.seed(CNN_RANDOM_STATE)

cnn_model = create_cnn_model(
    input_shape=CNN_INPUT_SHAPE,
    num_classes=len(class_names),
    learning_rate=CNN_LEARNING_RATE
)

cnn_model

## Training

The CNN is trained on the train split and monitored on the validation split.

Training uses batch loading through `DataLoader`; only batches are moved to GPU when CUDA is available. Early stopping restores the best validation-loss weights before evaluation.

In [ ]:
trained_cnn_model, training_history = train_torch_model(
    model=cnn_model,
    X_train=X_train,
    y_train=y_train,
    X_validation=X_validation,
    y_validation=y_validation,
    batch_size=CNN_BATCH_SIZE,
    epochs=CNN_EPOCHS,
    learning_rate=CNN_LEARNING_RATE,
    patience=5
)

history = pd.DataFrame(training_history)
history.tail()

## Training History Visualization

Training and validation curves help identify underfitting and overfitting. A widening gap between training and validation curves usually indicates overfitting.

In [ ]:
fig_history, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history["epoch"], history["train_loss"], label="train")
axes[0].plot(history["epoch"], history["validation_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history["epoch"], history["train_accuracy"], label="train")
axes[1].plot(history["epoch"], history["validation_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## Validation Evaluation

Validation metrics are used to understand model behavior before final test evaluation. Accuracy and macro F1 should be considered together because accuracy can be dominated by frequent genres.

In [ ]:
train_result = evaluate_torch_model(
    trained_cnn_model,
    X_train,
    y_train,
    class_names,
    model_name="cnn_baseline",
    split="train",
    batch_size=CNN_BATCH_SIZE
)

validation_result = evaluate_torch_model(
    trained_cnn_model,
    X_validation,
    y_validation,
    class_names,
    model_name="cnn_baseline",
    split="validation",
    batch_size=CNN_BATCH_SIZE
)

metrics_summary = pd.DataFrame([
    train_result["summary"],
    validation_result["summary"],
])

metrics_summary

In [ ]:
pd.DataFrame(validation_result["classification_report"]).transpose()

## Test Evaluation

The test split is evaluated once after training and validation review. This gives the final CNN baseline result for the current combined GTZAN + FMA train / validation / test strategy.

In [ ]:
test_result = evaluate_torch_model(
    trained_cnn_model,
    X_test,
    y_test,
    class_names,
    model_name="cnn_baseline",
    split="test",
    batch_size=CNN_BATCH_SIZE
)

test_summary = pd.DataFrame([test_result["summary"]])
test_summary

In [ ]:
test_report = pd.DataFrame(test_result["classification_report"]).transpose()
test_report

In [ ]:
fig_confusion, ax = plt.subplots(figsize=(9, 7))

display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=test_result["confusion_matrix"],
    display_labels=class_names
)

display_matrix.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title("Test Confusion Matrix - CNN Baseline")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## Result Interpretation

The CNN uses richer time-frequency input than MFCC summary vectors, so it can learn local spectral and temporal patterns directly from Mel Spectrograms.

This also makes training slower than classical Machine Learning because the model processes two-dimensional tensors instead of compact 40-dimensional MFCC vectors.

Accuracy and macro F1 should both be reviewed. If accuracy is stronger than macro F1, the model may be performing better on frequent genres while struggling with minority genres.

The classification report and confusion matrix should be used to inspect per-genre behavior rather than relying on a single aggregate metric.

## Save Model and Artifacts

The trained CNN model, label encoder, training history, metric summaries, test classification report, and test confusion matrix are saved under the configured CNN artifact paths.

Generated model and output artifacts are ignored by the existing `.gitignore` rules.

In [ ]:
CNN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CNN_MODEL_DIR.mkdir(parents=True, exist_ok=True)

save_torch_model(
    trained_cnn_model,
    BEST_CNN_MODEL_PATH
)

save_model(
    label_encoder,
    CNN_LABEL_ENCODER_PATH
)

history.to_csv(
    CNN_TRAINING_HISTORY_PATH,
    index=False
)

final_metrics_summary = pd.concat(
    [metrics_summary, test_summary],
    ignore_index=True
)

final_metrics_summary.to_csv(
    CNN_METRICS_SUMMARY_PATH,
    index=False
)

test_report_to_save = test_report.copy()
test_report_to_save.insert(0, "label", test_report_to_save.index)
test_report_to_save.reset_index(drop=True).to_csv(
    CNN_TEST_REPORT_PATH,
    index=False
)

fig_confusion.savefig(
    CNN_TEST_CONFUSION_MATRIX_PATH,
    dpi=150,
    bbox_inches="tight"
)

print("Saved CNN model:", BEST_CNN_MODEL_PATH)
print("Saved label encoder:", CNN_LABEL_ENCODER_PATH)
print("Saved training history:", CNN_TRAINING_HISTORY_PATH)
print("Saved metrics summary:", CNN_METRICS_SUMMARY_PATH)
print("Saved test report:", CNN_TEST_REPORT_PATH)
print("Saved test confusion matrix:", CNN_TEST_CONFUSION_MATRIX_PATH)

## Conclusion

This notebook trained and evaluated a CNN baseline using saved Mel Spectrogram tensors generated by Notebook 03.

The completed workflow includes:

- loading saved Mel Spectrogram train / validation / test tensors,
- encoding labels consistently,
- training a small CNN baseline,
- evaluating validation and test performance,
- reporting accuracy, macro F1, weighted F1, classification report, and confusion matrix,
- saving model and evaluation artifacts.

This keeps feature extraction separate from model training and avoids recomputing expensive Mel Spectrograms every time the CNN notebook is run.